# 02 – Data Preprocessing

**Projekt:** Nike / Adidas / Converse cipőmárkák képosztályozása  
**Cél:** az EDA során megvizsgált képek ellenőrzése, tisztítása, felosztása és a modellek számára előkészítése.

Ebben a Notebookban még nem tanítunk neurális hálózatot. A cél egy reprodukálható, dokumentált adat-előkészítési folyamat létrehozása.

**Bemenet:** `../dataset/brand/{nike,adidas,converse}/`  
**Kimenet:** `../dataset/brand_split/{train,validation,test}/{nike,adidas,converse}/`

> Fontos: a pontosan azonos fájlok kiszűrése automatizálható. Ugyanannak a cipőmodellnek különböző fotóit viszont csak akkor tudjuk biztosan egy csoportban tartani, ha ismerjük, mely képek tartoznak ugyanahhoz a termékhez. Ezt a felosztás előtt manuálisan is ellenőrizni kell.

## 1. Könyvtárak és reprodukálhatóság

Rögzítjük a véletlenmagot, hogy a felosztás és a mintavételezés megismételhető legyen. A képek betöltéséhez Pillow-t, az adatok kezeléséhez Pandas-t, a felosztáshoz Scikit-learn-t használunk.

In [ ]:
from pathlib import Path
import hashlib
import shutil
import random
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageOps, UnidentifiedImageError
from sklearn.model_selection import train_test_split
from IPython.display import display

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 2. Útvonalak és paraméterek

A korábbi EDA Notebook a `../dataset/brand` útvonalat használta, ezért ezt megtartjuk. Ha a mappastruktúrád eltér, csak a `DATASET_PATH` értékét kell módosítani.

A feldolgozott felosztás külön mappába kerül; az eredeti képeket nem mozgatjuk és nem írjuk felül.

In [ ]:
# A Notebook a notebooks/ mappában legyen.
PROJECT_ROOT = Path("..").resolve()

DATASET_PATH = PROJECT_ROOT / "dataset" / "brand"
OUTPUT_PATH = PROJECT_ROOT / "dataset" / "brand_split"
REPORT_PATH = PROJECT_ROOT / "results" / "brand"

CLASSES = ["nike", "adidas", "converse"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

IMG_SIZE = (224, 224)
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15

assert abs(TRAIN_RATIO + VALIDATION_RATIO + TEST_RATIO - 1.0) < 1e-9

print("Projekt gyökérkönyvtár:", PROJECT_ROOT)
print("Bemeneti adatok:", DATASET_PATH)
print("Kimeneti adatok:", OUTPUT_PATH)

missing_folders = [str(DATASET_PATH / c) for c in CLASSES if not (DATASET_PATH / c).exists()]
if missing_folders:
    raise FileNotFoundError(
        "Hiányzó osztálymappák. Ellenőrizd az EDA-ban használt útvonalat és a mappaneveket:\n"
        + "\n".join(missing_folders)
    )

## 3. Képfájlok összegyűjtése és ellenőrzése

Minden támogatott képfájlt beolvasunk. Ellenőrizzük, hogy a fájl ténylegesen megnyitható-e, és rögzítjük a méretét, formátumát, valamint a fájlméretét.

A sérült vagy nem olvasható képek nem kerülnek be a későbbi tanítóhalmazba.

In [ ]:
records = []
invalid_files = []

for label in CLASSES:
    class_dir = DATASET_PATH / label

    for path in sorted(class_dir.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        try:
            with Image.open(path) as img:
                img.verify()

            # Újranyitás szükséges a metaadatok biztonságos lekéréséhez.
            with Image.open(path) as img:
                width, height = img.size
                image_format = img.format

            records.append({
                "path": str(path.resolve()),
                "label": label,
                "width": width,
                "height": height,
                "format": image_format,
                "file_size_bytes": path.stat().st_size
            })

        except (UnidentifiedImageError, OSError, ValueError) as exc:
            invalid_files.append({
                "path": str(path.resolve()),
                "label": label,
                "error": str(exc)
            })

data = pd.DataFrame(records)
invalid_df = pd.DataFrame(invalid_files)

print(f"Érvényes képek: {len(data)}")
print(f"Hibás / nem olvasható képek: {len(invalid_df)}")
display(data.groupby("label").size().rename("valid_images").to_frame())

if not invalid_df.empty:
    display(invalid_df)


## 4. Osztályeloszlás ellenőrzése

Megnézzük, hogy a három márka képszáma továbbra is közel azonos-e. Az osztályok közötti nagy eltérés torzíthatja a modellt, ezért ezt a tanítás előtt dokumentálni kell.

In [ ]:
class_counts = (
    data["label"]
    .value_counts()
    .reindex(CLASSES, fill_value=0)
    .rename_axis("brand")
    .reset_index(name="image_count")
)

display(class_counts)

plt.figure(figsize=(8, 5))
sns.barplot(data=class_counts, x="brand", y="image_count")
plt.title("Érvényes képek száma márkánként")
plt.xlabel("Márka")
plt.ylabel("Képek száma")
plt.tight_layout()
plt.show()

if (class_counts["image_count"] == 0).any():
    raise ValueError("Legalább egy osztályban nincs érvényes kép. Ellenőrizd a mappákat.")

## 5. Pontosan azonos képek keresése

SHA-256 hash segítségével felismerjük azokat a fájlokat, amelyek tartalma bájtról bájtra megegyezik, akkor is, ha más fájlnévvel szerepelnek.

- Ha egy azonos kép ugyanazzal a címkével többször szerepel, csak egy példányt tartunk meg.
- Ha ugyanaz a képfájl több különböző címkéhez tartozik, azt címkézési konfliktusként jelezzük, és az összes érintett példányt kizárjuk az automatikus felosztásból.

Ez nem találja meg az ugyanarról a cipőről készült, de eltérő fotókat.

In [ ]:
def sha256_file(file_path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(file_path, "rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

data["sha256"] = data["path"].apply(sha256_file)

hash_label_counts = data.groupby("sha256")["label"].nunique()
conflict_hashes = set(hash_label_counts[hash_label_counts > 1].index)

conflicts = data[data["sha256"].isin(conflict_hashes)].sort_values("sha256")
if not conflicts.empty:
    print("FIGYELEM: azonos tartalmú kép több címkével is szerepel:")
    display(conflicts[["path", "label", "sha256"]])

# A különböző címkékkel előforduló hash-ek minden példányát kizárjuk.
without_conflicts = data[~data["sha256"].isin(conflict_hashes)].copy()

# Az azonos címkéhez tartozó bájtszinten duplikált képekből egy példány marad.
deduplicated = without_conflicts.drop_duplicates(subset="sha256", keep="first").copy()

print("Kiinduló érvényes képek:", len(data))
print("Címkézési konfliktus miatt kizárt képek:", len(data) - len(without_conflicts))
print("Duplikátumok eltávolítása után:", len(deduplicated))
print("Eltávolított azonos duplikátumok:", len(without_conflicts) - len(deduplicated))

data = deduplicated.reset_index(drop=True)

if data.empty:
    raise ValueError("A tisztítás után nem maradt feldolgozható kép.")

## 6. A felosztás előtti manuális ellenőrzés

A legfontosabb adatszivárgási kockázat az, ha ugyanannak a konkrét cipőnek több fotója különböző részhalmazokba kerül. A hash-ellenőrzés csak a pontosan azonos fájlokat találja meg.

**Javasolt ellenőrzés:** nézd át a képeket márkánként, és azonosítsd az ugyanarról a cipőről vagy termékoldalról származó képcsoportokat. Ha ilyenek vannak, készíts egy `product_groups.csv` fájlt az alábbi oszlopokkal:

`path, product_group`

A `path` oszlopban a képek teljes elérési útja szerepeljen, a `product_group` pedig az azonos cipőmodellhez tartozó közös azonosító legyen. Ha nincs ilyen információd, a következő felosztás csak a pontos fájl-duplikátumok ellen véd; a termékszintű adatszivárgás lehetősége megmarad.

In [ ]:
# Opcionális termékcsoport-fájl.
# Ha nincs ilyen fájlod, a cella minden képet önálló csoportként kezel.
GROUPS_FILE = PROJECT_ROOT / "dataset" / "product_groups.csv"

if GROUPS_FILE.exists():
    groups_df = pd.read_csv(GROUPS_FILE)

    required_columns = {"path", "product_group"}
    if not required_columns.issubset(groups_df.columns):
        raise ValueError(f"A product_groups.csv oszlopai legyenek: {required_columns}")

    groups_df["path"] = groups_df["path"].map(lambda p: str(Path(p).resolve()))
    data = data.merge(groups_df, on="path", how="left")

    missing_groups = data["product_group"].isna().sum()
    if missing_groups:
        print(f"Figyelem: {missing_groups} képhez nincs termékcsoport. Ezeket egyedi csoportként kezeljük.")

    data["group_id"] = data["product_group"].fillna(data["sha256"])
else:
    print("Nincs product_groups.csv. A csoportosítás csak a pontos duplikátumok kiszűrésére támaszkodik.")
    data["group_id"] = data["sha256"]

# Egy csoporthoz csak egy címke tartozhat.
group_label_counts = data.groupby("group_id")["label"].nunique()
bad_groups = set(group_label_counts[group_label_counts > 1].index)

if bad_groups:
    raise ValueError(
        "Van olyan product_group, amely több márkához is tartozik. "
        "Javítsd a product_groups.csv címkéit."
    )

print("Feldolgozható képek:", len(data))
print("Csoportok száma:", data["group_id"].nunique())

## 7. Train / Validation / Test felosztás

A cél 70% tanító-, 15% validációs és 15% tesztadat.

A felosztás két lépésben történik, a termékcsoportokat együtt tartva. Így egy megadott termékcsoport nem kerülhet egyszerre például a tanító- és a teszthalmazba.

A csoportos felosztás miatt a pontos arányok kissé eltérhetnek a célértékektől. Ez jobb kompromisszum, mint az azonos termékhez tartozó képek szétosztása. Az osztályeloszlást az alábbi táblázatban ellenőrizzük.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Csoportos véletlen felosztás. A csoportok egyben maradnak.
# A GroupShuffleSplit nem garantál pontos osztályarányokat; ezeket ellenőrizzük utána.
first_split = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_RATIO,
    random_state=RANDOM_STATE
)

train_val_idx, test_idx = next(
    first_split.split(data, y=data["label"], groups=data["group_id"])
)

train_val = data.iloc[train_val_idx].copy()
test = data.iloc[test_idx].copy()

relative_val_size = VALIDATION_RATIO / (TRAIN_RATIO + VALIDATION_RATIO)

second_split = GroupShuffleSplit(
    n_splits=1,
    test_size=relative_val_size,
    random_state=RANDOM_STATE + 1
)

train_idx, val_idx = next(
    second_split.split(
        train_val,
        y=train_val["label"],
        groups=train_val["group_id"]
    )
)

train = train_val.iloc[train_idx].copy()
validation = train_val.iloc[val_idx].copy()

# Ellenőrzés: termékcsoportok nem kerülhetnek több részhalmazba.
train_groups = set(train["group_id"])
val_groups = set(validation["group_id"])
test_groups = set(test["group_id"])

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

print("A felosztás elkészült.")
print(f"Train: {len(train)} ({len(train)/len(data):.1%})")
print(f"Validation: {len(validation)} ({len(validation)/len(data):.1%})")
print(f"Test: {len(test)} ({len(test)/len(data):.1%})")

## 8. A felosztás ellenőrzése

Nem elég csak a teljes darabszámot nézni: ellenőrizzük a márkák eloszlását is mindhárom részhalmazban. A csoportos felosztás miatt előfordulhat kisebb eltérés a 70/15/15 aránytól és az osztályarányoktól.

In [ ]:
splits = {
    "train": train,
    "validation": validation,
    "test": test
}

distribution = pd.DataFrame({
    name: split["label"].value_counts().reindex(CLASSES, fill_value=0)
    for name, split in splits.items()
}).T

distribution["total"] = distribution.sum(axis=1)
display(distribution)

distribution.plot(
    kind="bar",
    stacked=True,
    figsize=(9, 5)
)
plt.title("Márkák eloszlása a részhalmazokban")
plt.xlabel("Adathalmaz")
plt.ylabel("Képek száma")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Figyelmeztetés, ha valamelyik osztály teljesen hiányzik valamelyik részhalmazból.
for split_name, split in splits.items():
    missing_classes = set(CLASSES) - set(split["label"].unique())
    if missing_classes:
        print(f"FIGYELEM: {split_name} részhalmazból hiányzik: {sorted(missing_classes)}")

## 9. Képek átméretezése és mentése mappastruktúrába

A képeket 224 × 224 pixelre méretezzük, RGB formátumra alakítjuk, majd külön train, validation és test mappákba másoljuk.

**Miért nem normalizálunk itt 0–1 közé?** A választott MobileNetV2 előfeldolgozó függvénye saját skálázást végez (a pixeleket a modell által elvárt tartományba alakítja). A saját CNN-hez később külön, megfelelő előfeldolgozást használunk. Az eredeti képek megmaradnak a `dataset/brand` mappában.

In [ ]:
def prepare_and_save_image(source_path, destination_path, size=IMG_SIZE):
    destination_path.parent.mkdir(parents=True, exist_ok=True)

    with Image.open(source_path) as img:
        img = ImageOps.exif_transpose(img)
        img = img.convert("RGB")
        img = img.resize(size, Image.Resampling.LANCZOS)
        img.save(destination_path, format="JPEG", quality=95, optimize=True)

# Csak az általunk kezelt kimeneti mappát töröljük újragenerálás előtt.
# Ellenőrizd az OUTPUT_PATH értékét, mielőtt ezt a cellát futtatod.
if OUTPUT_PATH.exists():
    shutil.rmtree(OUTPUT_PATH)

for split_name, split_df in splits.items():
    for row in split_df.itertuples(index=False):
        source = Path(row.path)
        destination = OUTPUT_PATH / split_name / row.label / f"{source.stem}.jpg"
        prepare_and_save_image(source, destination)

print("Előkészített képek mentve ide:", OUTPUT_PATH)

## 10. Split-manifestek és osztálynevek mentése

A fájlok másolata mellett CSV-manifesteket is mentünk. Ezek dokumentálják, hogy melyik eredeti kép melyik részhalmazba került. A `class_names.json` rögzíti a címkék sorrendjét, hogy a későbbi predikció során ne legyen osztálysorrendből eredő hiba.

In [ ]:
REPORT_PATH.mkdir(parents=True, exist_ok=True)

for split_name, split_df in splits.items():
    manifest = split_df.copy()
    manifest["split"] = split_name
    manifest.to_csv(
        REPORT_PATH / f"{split_name}_manifest.csv",
        index=False
    )

with open(REPORT_PATH / "class_names.json", "w", encoding="utf-8") as file:
    json.dump(CLASSES, file, ensure_ascii=False, indent=4)

summary = {
    "random_state": RANDOM_STATE,
    "image_size": list(IMG_SIZE),
    "classes": CLASSES,
    "counts": {name: int(len(split)) for name, split in splits.items()},
    "total_images": int(len(data))
}

with open(REPORT_PATH / "preprocessing_summary.json", "w", encoding="utf-8") as file:
    json.dump(summary, file, ensure_ascii=False, indent=4)

print("Manifestek és összefoglaló mentve ide:", REPORT_PATH)

## 11. Végső ellenőrzés

Ellenőrizzük, hogy minden részhalmazban létrejöttek-e az osztálymappák, és a fájlok száma megegyezik-e a manifestekben szereplő darabszámmal.

In [ ]:
for split_name, split_df in splits.items():
    saved_count = sum(
        1
        for p in (OUTPUT_PATH / split_name).rglob("*")
        if p.is_file() and p.suffix.lower() == ".jpg"
    )

    print(
        f"{split_name:12s} | "
        f"manifest: {len(split_df):4d} | "
        f"mentett képek: {saved_count:4d}"
    )

    assert saved_count == len(split_df), (
        f"A mentett képek száma nem egyezik a {split_name} manifesttel."
    )

print("\nData preprocessing sikeresen befejeződött.")

## 12. Következő lépés

A következő Notebookban (`03_Model_Training.ipynb`) a `dataset/brand_split/` mappából töltjük be a képeket, majd:

1. létrehozunk egy saját CNN modellt;
2. betanítjuk a MobileNetV2 Transfer Learning modellt;
3. összehasonlítjuk a két modell teljesítményét;
4. a teszthalmazon kiszámítjuk az Accuracy, Precision, Recall és F1-score értékeket.

**Fontos korlát:** ha nem áll rendelkezésre termékszintű csoportosítás, akkor a jelenlegi csoportos felosztás csak az általunk azonosított csoportokat tartja együtt. A különböző fotókon szereplő, de azonos cipőmodellből eredő adatszivárgást manuális ellenőrzéssel lehet tovább csökkenteni.